# DroneFuse - F1: every figure of the paper from the saved experiment outputs (CPU)

This notebook draws the figures of the article. It **trains nothing new and types no number in**: every figure
reads the files that the earlier Kaggle runs saved. Each figure is written as a vector PDF (for the paper), a
PNG (to look at) and a CSV with exactly the values plotted. `figures_manifest.json` lists, for every figure, the
input files it read (with their SHA-256) and the checks it passed. Where a figure needs per-window predictions
that a run did not keep (DroneRFc-MM), the T3 model is refitted from the cached features, and the figure is
drawn only if the refit reproduces the saved accuracy and AUROC.

**Kaggle settings:** Accelerator = **None (CPU)**. Internet = Off is fine.

**Inputs to add** (a figure whose input is missing is skipped, the others are still made):

| Input | Gives |
|---|---|
| `dronefuse-code` (latest) | the code and the fixed splits (`splits/windows.csv`) - **required** |
| `dronefuse-step3-results` (Step 3 output with `runs/` and `runs_crop/`) | accuracy intervals, training curves, confusion matrices, leave-one-out AUROC |
| `dronefuse-step2-features` and `dronefuse-step2b-crops` | MMAUD probes and embedding maps |
| `dronefuse-mmaud-v2b` (Step 1 output holding `V2_p4s1`) | MMAUD sensor examples |
| T3 output(s) saved as datasets (`t3/t3_results.json` + `t3/cache/`); if DroneRFc-MM and TSMS-Drone came from two T3 runs, attach both | DroneRFc-MM and TSMS-Drone figures |
| `dronefuse-rfcmm-core` (T2) and `dronefuse-tsms` (T1) | DroneRFc-MM and TSMS-Drone sensor examples |

**Run time:** about 20-30 min (most of it the 1000-replicate block bootstrap and the t-SNE maps).

**Output:** `f1/` with the figures, `figures_manifest.json`, `f1_report.txt`, `mmaud_stats.json`, and
`f1_figures.zip` (everything in one file to download).

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/f1')
N_BOOT = 1000                    # block-bootstrap replicates (as in the paper)
MMAUD_EXAMPLE = 'V2_p4s1'        # recording shown in the MMAUD sensor-example figure
RFCMM_SESSION = 'G1'             # DroneRFc-MM session shown in the example figure
TSMS_DISTANCE = 10               # TSMS-Drone distance (m) shown in the example figure
TIME_BUDGET_MINUTES = 11 * 60

In [ ]:
import hashlib, json, shutil
import numpy as np
from dronefuse import environment, figures as fg
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'environment': runner.run('measure the machine', environment.collect)}
notes = []

def log(*parts):
    text = ' '.join(str(p) for p in parts)
    print(text)
    notes.append(text)

# The code version: one hash over every module of the package, written into the manifest.
package_dir = Path(dronefuse.__file__).parent
code_version = hashlib.sha256(b''.join(p.read_bytes() for p in sorted(package_dir.glob('*.py')))).hexdigest()[:16]
found = fg.find_inputs(INPUT, code_root)
for line in fg.describe_inputs(found):
    log(line)
log('code version', code_version)
F = fg.Figures(OUTPUT, code_version=code_version)
WINDOWS = found['windows_csv']
STEP3 = found['step3']
t3, t3_inputs = fg.load_t3(found)          # every attached T3 output, merged per dataset
log('T3 results for:', sorted(t3) or 'none')
res = None
mm_probes = None

In [ ]:
# MMAUD: the recordings, and every figure that comes from the saved Step 3 runs.
runner.run('Figure: MMAUD recordings and time blocks', fg.fig_timeline, F, WINDOWS)
if STEP3:
    res = runner.run('MMAUD block bootstrap from the saved runs', fg.mmaud_stats, STEP3, WINDOWS, N_BOOT,
                     OUTPUT / 'mmaud_stats.json')
    if res:
        runner.run('Figure: same vs new flights (MMAUD)', fg.fig_within_vs_cross_mmaud, F, res,
                   [WINDOWS, OUTPUT / 'mmaud_stats.json'])
    runner.run('Figure: training curves', fg.fig_training_curves, F, STEP3)
    runner.run('Figure: confusion matrices (MMAUD)', fg.fig_confusion_mmaud, F, STEP3, WINDOWS)
else:
    log('Step 3 results not attached: MMAUD result figures skipped')

In [ ]:
# MMAUD: probes and embedding maps from the Step 2 (and 2b) features.
if found['step2']:
    mm_probes = runner.run('MMAUD probes (type and recording)', fg.mmaud_probes, found['step2'], WINDOWS,
                           found['step2b'])
    if mm_probes:
        (OUTPUT / 'mmaud_probes.json').write_text(json.dumps(mm_probes, indent=1))
    runner.run('Figure: embedding maps (MMAUD)', fg.fig_embedding_mmaud, F, found['step2'], WINDOWS)
else:
    log('Step 2 features not attached: MMAUD probes and embedding maps skipped')

# MMAUD: one window of every sensor, read from the Step 1 extract.
if MMAUD_EXAMPLE in found['step1']:
    window = None
    if found['step2']:
        window = runner.run('choose the example window', fg.pick_mmaud_window, found['step2'], MMAUD_EXAMPLE)
    runner.run('Figure: MMAUD sensor examples', fg.fig_examples_mmaud, F, found['step1'][MMAUD_EXAMPLE], WINDOWS,
               window)
else:
    log(f'Step 1 extract of {MMAUD_EXAMPLE} not attached: MMAUD sensor examples skipped')

In [ ]:
# DroneRFc-MM and TSMS-Drone: results saved by T3, and the refits from its feature cache.
if t3:
    runner.run('Figure: same vs new sessions (DroneRFc-MM, TSMS-Drone)', fg.fig_within_vs_cross_more, F, t3,
               t3_inputs)
if t3 or mm_probes:
    runner.run('Figure: probes on the three datasets', fg.fig_probes, F, mm_probes, t3,
               t3_inputs + ([found['step2']] if mm_probes else []))
if res or t3:
    runner.run('Figure: leave-one-out AUROC maps', fg.fig_loo_auroc, F, res, t3,
               t3_inputs + ([OUTPUT / 'mmaud_stats.json'] if res else []))

if found['t3_cache'] and (Path(found['t3_cache']) / 'rfcmm_features.pkl').exists() and 'rfcmm' in t3:
    loaded = runner.run('load the DroneRFc-MM feature cache', fg.load_rfcmm_cache, found['t3_cache'])
    if loaded:
        rf, rf_path = loaded
        refit = runner.run('refit the T3 models and check them against the saved results', fg.rfcmm_refit, rf,
                           t3['rfcmm'])
        if refit:
            refit, checks = refit
            for c in checks:
                log('  ' + c)
            runner.run('Figure: confusion matrices (DroneRFc-MM)', fg.fig_confusion_rfcmm, F, refit, checks,
                       [rf_path] + t3_inputs)
            runner.run('Figure: unknown-type scores (DroneRFc-MM)', fg.fig_scores_rfcmm, F, refit, checks,
                       [rf_path] + t3_inputs)
        runner.run('Figure: embedding maps (DroneRFc-MM)', fg.fig_embedding_rfcmm, F, rf, rf_path)
else:
    log('T3 DroneRFc-MM feature cache not attached: its confusion, score and embedding figures skipped')
if found['t3_cache_tsms']:
    runner.run('Figure: embedding maps (TSMS-Drone)', fg.fig_embedding_tsms, F, found['t3_cache_tsms'])
else:
    log('T3 TSMS feature cache not attached: TSMS embedding maps skipped')

# Raw sensor examples of the two further datasets.
if found['rfcmm_root'] or found['tsms_packs']:
    runner.run('Figure: DroneRFc-MM and TSMS-Drone sensor examples', fg.fig_examples_more, F,
               found['rfcmm_root'], found['tsms_packs'] or None, RFCMM_SESSION, TSMS_DISTANCE)
else:
    log('raw DroneRFc-MM / TSMS data not attached: their sensor examples skipped')

In [ ]:
# Report, manifest and one zip with everything
report = fg.format_report(found, F, runner, notes)
(OUTPUT / 'f1_report.txt').write_text(report)
print(report)
archive = shutil.make_archive('/kaggle/working/f1_figures', 'zip', OUTPUT)
shutil.move(archive, OUTPUT / 'f1_figures.zip')
print('download:', OUTPUT / 'f1_figures.zip')

## After the run

Download `f1/f1_figures.zip` (Output tab) and send it back, or at least `f1_report.txt` and
`figures_manifest.json`. The PDFs go into `paper/figures/`; each figure's CSV holds the plotted values.